In [1]:
import argparse
import os
import sys
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler, normalize # For L2 normalization
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score # For Clustering Evaulation

import plotly.express as px

import json

In [2]:
def parse_args():
    defaults = {
        "input_dir": "../../data/embedded",
        "output_dir": "../../data/results",
        "generate_plot": True,
        "algorithm": "kmeans",
        "n_clusters": 5,
        "eps": 10.0,
        "min_samples": 3,
        "reducer": "pca",
        "perplexity": 30.0
    }
    return defaults

In [3]:
def load_data_recursive(input_dir):
    embeddings_list = []
    filenames = []

    print(f"Scanning '{input_dir}' for embeddings...")

    files_found = 0
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.endswith(".npy"):
                file_path = os.path.join(root, file)

                try:
                    data = np.load(file_path)

                    if data.ndim == 0:
                        continue
                    elif data.ndim == 2:
                        data = data.squeeze()

                    if data.shape != (1024,):
                        continue

                    embeddings_list.append(data)
                    filenames.append(file.replace(".npy", ""))
                    files_found += 1

                except Exception as e:
                    print(f"Error loading {file}: {e}")

    if files_found == 0:
        return None, None

    print(f"Found {files_found} valid files.")

    X = np.vstack(embeddings_list)
    return X, filenames

In [4]:
def main():
    args = parse_args()

    # LOAD DATA
    if not os.path.exists(args['input_dir']):
        print(f"Error: Directory '{args['input_dir']}' not found.")
        sys.exit(0)

    os.makedirs(args['output_dir'], exist_ok=True)

    X, file_labels = load_data_recursive(args['input_dir'])

    if X is None:
        print("No valid embeddings found. Check your directory.")
        sys.exit(0)

    print(f"Final Matrix Shape: {X.shape} (Samples: {X.shape[0]}, Features: {X.shape[1]})")

    # PREPROCESSING
    print("Scaling features...")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    ## Also reduce dimensionality first
    print("Reducing dimensionality before clustering...")
    reducer_before_clustering = PCA(n_components = 50)
    X_reduced_scaled = reducer_before_clustering.fit_transform(X_scaled)

    # CLUSTERING
    print(f"Running Clustering ({args['algorithm']})...")

    if args['algorithm'] == "kmeans":
        print(f"KMeans with {args['n_clusters']} clusters")
        model = KMeans(n_clusters=args['n_clusters'], random_state=42, n_init=10)
        X_normalized_reduced_scaled = normalize(X_reduced_scaled, norm="l2")
        labels = model.fit_predict(X_normalized_reduced_scaled)

    elif args['algorithm'] == "dbscan":
        print(f"DBSCAN with eps={args['eps']}, min_samples={args['min_samples']}")
        model = DBSCAN(eps=args['eps'], min_samples=3)
        labels = model.fit_predict(X_reduced_scaled)

    # DIMENSIONALITY REDUCTION
    print(f"Reducing dimensions using {args['reducer'].upper()}...")

    if args['reducer'] == "pca":
        reducer = PCA(n_components=2)
        X_2d = reducer.fit_transform(X_reduced_scaled)

    elif args['reducer'] == "tsne":
        pca_50 = PCA(n_components=min(50, X.shape[1]))
        X_pca = pca_50.fit_transform(X_reduced_scaled)

        tsne = TSNE(n_components=2, perplexity=args['perplexity'], random_state=42, init='pca', learning_rate='auto')
        X_2d = tsne.fit_transform(X_pca)

    # VISUALIZATION
    df_plot = pd.DataFrame(X_2d, columns=['Component 1', 'Component 2'])

    df_plot['Cluster'] = labels.astype(str) 
    df_plot['Source'] = file_labels

    df_plot['ecg_id'] = df_plot['Source'].astype(int)

    try:
        true_labels_df = pd.read_csv('../../data/preprocessed/cleaned_labels.csv')
        df_plot = df_plot.merge(true_labels_df[['ecg_id', 'label']], on='ecg_id', how='left')
        df_plot['label'] = df_plot['label'].fillna('Unknown')
    except Exception as e:
        print(f"Could not load true labels: {e}")
        df_plot['label'] = 'Unknown'
    
    if args['generate_plot']:
        print("Generating Interactive Plot...")

        title = f"MOMENT Embeddings: {args['reducer'].upper()} Projection<br>({args['algorithm'].capitalize()})"

        fig = px.scatter(
            df_plot,
            x='Component 1',
            y='Component 2',
            color='label',
            symbol='Cluster',
            hover_name='Source',
            hover_data={'label': True, 'Cluster': True, 'Component 1': False, 'Component 2': False},
            title=title,
            labels={
                'Component 1': f"{args['reducer'].upper()} Dimension 1",
                'Component 2': f"{args['reducer'].upper()} Dimension 2"
            },
            color_discrete_sequence=px.colors.qualitative.Plotly
        )

        fig.update_traces(marker=dict(size=10, line=dict(width=1, color='DarkSlateGrey')), opacity=0.8)
        fig.update_layout(template="plotly_white")
        
        filename = f"plot_{args['algorithm']}_{args['reducer']}.html"
        save_path = os.path.join(args['output_dir'], filename)

        fig.write_html(save_path)
        print(f"Interactive plot saved to: {save_path}")

    # DATA JSON
    print("Exporting raw data for GUI...")
    
    json_save_path = os.path.join(args['output_dir'], "real_chart_data.json")
    chart_data = []

    for idx, row in df_plot.iterrows():
        chart_data.append({
            "x": row['Component 1'],
            "y": row['Component 2'],
            "cluster": row['Cluster'],
            "source": row['Source'],
            "label": row['label']
        })

    with open(json_save_path, 'w') as f:
        json.dump(chart_data, f, indent=4)

    print(f"GUI data saved to: {json_save_path}")

    # --- CLUSTERING EVALUATION ---
    print("\n" + "="*40)
    print("CLUSTERING METRICS")
    print("="*40)

    # Silhouette Score
    sil_score = silhouette_score(X_normalized_reduced_scaled, labels)
    print(f"Silhouette Score: {sil_score:.4f} (Closer to 1 = better separated)")

    # External Metrics
    valid_idx = df_plot['label'] != 'Unknown'
    true_labels_valid = df_plot.loc[valid_idx, 'label']
    pred_clusters_valid = df_plot.loc[valid_idx, 'Cluster']

    if len(true_labels_valid) > 0:
        # Calculate ARI and NMI
        ari = adjusted_rand_score(true_labels_valid, pred_clusters_valid)
        nmi = normalized_mutual_info_score(true_labels_valid, pred_clusters_valid)
        
        print(f"Adjusted Rand Index (ARI): {ari:.4f} (Closer to 1 = better match to true labels)")
        print(f"Normalized Mutual Info (NMI):  {nmi:.4f} (Closer to 1 = more information shared)")
    else:
        print("Could not calculate ARI/NMI: No true labels found.")
    
    print("="*40 + "\n")

In [5]:
main()

Scanning '../../data/embedded' for embeddings...
Found 16272 valid files.
Final Matrix Shape: (16272, 1024) (Samples: 16272, Features: 1024)
Scaling features...
Reducing dimensionality before clustering...
Running Clustering (kmeans)...
KMeans with 5 clusters
Reducing dimensions using PCA...
Generating Interactive Plot...
Interactive plot saved to: ../../data/results\plot_kmeans_pca.html
Exporting raw data for GUI...
GUI data saved to: ../../data/results\real_chart_data.json

CLUSTERING METRICS
Silhouette Score: 0.0986 (Closer to 1 = better separated)
Adjusted Rand Index (ARI): 0.0655 (Closer to 1 = better match to true labels)
Normalized Mutual Info (NMI):  0.0760 (Closer to 1 = more information shared)

